# Case Study — Meridian MMM & Digital Performance Stack
**Company (context):** D2B · **Synthetic data for portfolio demo**

Goal: estimate channel contribution / ROI and forecast leads with a lightweight model proxy.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error

rng = np.random.default_rng(7)
weeks = pd.date_range("2024-01-01", periods=78, freq="W-MON")
spend = pd.DataFrame({
    "week": weeks,
    "meta": rng.uniform(8, 25, len(weeks)) * 1000,
    "google": rng.uniform(10, 30, len(weeks)) * 1000,
    "tiktok": rng.uniform(3, 14, len(weeks)) * 1000,
    "seasonality": np.sin(np.linspace(0, 6*np.pi, len(weeks))),
})
# True-ish response curve for synthetic ground truth
true = (
    0.018 * spend["meta"]
    + 0.022 * spend["google"]
    + 0.015 * spend["tiktok"]
    + 1200 * spend["seasonality"]
    + rng.normal(0, 400, len(weeks))
)
spend["leads"] = np.clip(true, 200, None).astype(int)
spend["revenue"] = spend["leads"] * rng.uniform(28, 42, len(weeks))
spend.head()


In [ ]:
# Simple MMM proxy: Ridge on media spend
X = spend[["meta", "google", "tiktok", "seasonality"]]
y = spend["leads"]
model = Ridge(alpha=1.0)
model.fit(X, y)
coefs = pd.Series(model.coef_[:3], index=["meta", "google", "tiktok"])
contrib = (X[["meta", "google", "tiktok"]] * coefs.values).clip(lower=0)
share = contrib.sum() / contrib.sum().sum()
roi = (contrib.sum() * spend["revenue"].sum() / spend["leads"].sum()) / spend[["meta", "google", "tiktok"]].sum()
pd.DataFrame({"contribution_share": share.round(3), "roi_proxy": roi.round(2)})


In [ ]:
# Forecast last 8 weeks (holdout) with lag features
feat = spend.copy()
feat["leads_lag1"] = feat["leads"].shift(1)
feat["leads_lag2"] = feat["leads"].shift(2)
feat = feat.dropna()
split = len(feat) - 8
train, test = feat.iloc[:split], feat.iloc[split:]
cols = ["meta", "google", "tiktok", "seasonality", "leads_lag1", "leads_lag2"]
fmodel = Ridge(alpha=0.5)
fmodel.fit(train[cols], train["leads"])
pred = fmodel.predict(test[cols])
mape = mean_absolute_percentage_error(test["leads"], pred)
{"holdout_mape": round(float(mape), 3), "pred_mean_leads": int(pred.mean())}


In [ ]:
kpis = {
    "blended_roi_x": round(float(spend["revenue"].sum() / spend[["meta", "google", "tiktok"]].sum().sum()), 2),
    "avg_weekly_leads": int(spend["leads"].mean()),
    "best_channel_by_share": share.idxmax(),
}
kpis
